In [ ]:
!pip install fastapi uvicorn pyngrok -q

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

In [ ]:
!pip install groq -q

In [ ]:
!pip install transformers torch -q

In [ ]:
from transformers import pipeline
import json

# Load free model
generator = pipeline("text-generation", model="gpt2")

# Game state
player_state = {
    "health": 100,
    "inventory": [],
    "quest": "Find the lost sword of Eldoria"
}

def dungeon_master(user_input, player_state):
    prompt = f"""You are an AI Dungeon Master.
Player state: {json.dumps(player_state)}
Player says: {user_input}
Dungeon Master responds:"""

    response = generator(prompt, max_new_tokens=150, num_return_sequences=1)
    return response[0]['generated_text'].split("Dungeon Master responds:")[-1].strip()

# Test it
print("=== AI Dungeon Master ===")
print(f"Quest: {player_state['quest']}")
print(f"Health: {player_state['health']}")
print()

test_input = "I enter the dark forest and look around"
response = dungeon_master(test_input, player_state)
print(f"Player: {test_input}")
print(f"DM: {response}")

In [ ]:
!pip install -q transformers accelerate -q

In [ ]:
from transformers import pipeline
import json

# Small but good model - only 1.5GB
generator = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0")

player_state = {
    "health": 100,
    "inventory": ["torch", "dagger"],
    "quest": "Find the lost sword of Eldoria",
    "location": "village entrance"
}

conversation_history = []

def dungeon_master(user_input):
    conversation_history.append(f"Player: {user_input}")

    prompt = f"""<|system|>You are a creative AI Dungeon Master running a fantasy RPG.</s>
<|user|>Player state: {json.dumps(player_state)}
Player action: {user_input}</s>
<|assistant|>"""

    response = generator(prompt, max_new_tokens=150, do_sample=True, temperature=0.7)
    reply = response[0]['generated_text'].split("<|assistant|>")[-1].strip()
    conversation_history.append(f"DM: {reply}")
    return reply

print("=== AI Dungeon Master ===\n")
r1 = dungeon_master("I enter the dark forest and look around")
print(f"Player: I enter the dark forest and look around")
print(f"DM: {r1}\n")

In [ ]:
# Interactive mode
print("=== AI Dungeon Master - Interactive Mode ===")
print("Type 'quit' to exit\n")
print("DM: You stand at the entrance of the dark forest. Your quest: Find the lost sword of Eldoria. What do you do?\n")

while True:
    user_input = input("You: ")
    if user_input.lower() == "quit":
        print("Game over!")
        break
    response = dungeon_master(user_input)
    print(f"DM: {response}\n")

In [ ]:
import json
import re

# Updated player state
player_state = {
    "health": 100,
    "inventory": ["torch", "dagger"],
    "quest": "Find the lost sword of Eldoria",
    "location": "village entrance"
}

# New system prompt forcing JSON output
system_prompt = """You are an AI Dungeon Master running a fantasy RPG.
You MUST respond ONLY in valid JSON format like this:
{
    "narrative": "description of what happens",
    "npc_dialogue": "any NPC speech or empty string",
    "player_state": {
        "health": <number 0-100>,
        "inventory": ["item1", "item2"],
        "quest": "current quest",
        "location": "current location"
    },
    "choices": ["option 1", "option 2", "option 3"]
}
Never respond with plain text. Always valid JSON only."""

conversation_history = []

def dungeon_master_json(user_input, player_state):
    conversation_history.append(f"Player: {user_input}")

    prompt = f"""<|system|>{system_prompt}</s>
<|user|>Player state: {json.dumps(player_state)}
Player action: {user_input}</s>
<|assistant|>"""

    response = generator(prompt, max_new_tokens=200, do_sample=True, temperature=0.7)
    raw = response[0]['generated_text'].split("<|assistant|>")[-1].strip()

    # Try to parse JSON
    try:
        # Extract JSON from response
        json_match = re.search(r'\{.*\}', raw, re.DOTALL)
        if json_match:
            result = json.loads(json_match.group())
            # Update player state from response
            if "player_state" in result:
                player_state.update(result["player_state"])
            conversation_history.append(f"DM: {result.get('narrative', '')}")
            return result
        else:
            return {"narrative": raw, "choices": [], "player_state": player_state}
    except json.JSONDecodeError:
        return {"narrative": raw, "choices": [], "player_state": player_state}

# Test it
print("=== Testing JSON Output ===\n")
result = dungeon_master_json("I enter the dark forest and look around", player_state)
print("Raw JSON response:")
print(json.dumps(result, indent=2))
print(f"\nNarrative: {result.get('narrative', '')}")
print(f"Choices: {result.get('choices', [])}")
print(f"Player Health: {player_state['health']}")

In [ ]:
from groq import Groq
import json
import re

import os
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# Game state
player_state = {
    "health": 100,
    "inventory": ["torch", "dagger"],
    "quest": "Find the lost sword of Eldoria",
    "location": "village entrance"
}

conversation_history = []

system_prompt = """You are an AI Dungeon Master running a fantasy RPG.
You MUST respond ONLY in valid JSON format like this:
{
    "narrative": "description of what happens",
    "npc_dialogue": "any NPC speech or empty string",
    "player_state": {
        "health": <number 0-100>,
        "inventory": ["item1", "item2"],
        "quest": "current quest",
        "location": "current location"
    },
    "choices": ["option 1", "option 2", "option 3"]
}
Never respond with plain text. Always valid JSON only."""

def dungeon_master(user_input, player_state):
    conversation_history.append({"role": "user", "content": user_input})

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Player state: {json.dumps(player_state)}"},
            *conversation_history
        ],
        temperature=0.7,
        max_tokens=500
    )

    raw = response.choices[0].message.content.strip()
    conversation_history.append({"role": "assistant", "content": raw})

    try:
        json_match = re.search(r'\{.*\}', raw, re.DOTALL)
        if json_match:
            result = json.loads(json_match.group())
            if "player_state" in result:
                player_state.update(result["player_state"])
            return result
        else:
            return {"narrative": raw, "choices": [], "player_state": player_state}
    except:
        return {"narrative": raw, "choices": [], "player_state": player_state}

# Test
print("=== AI Dungeon Master (Groq LLaMA 70B) ===\n")
result = dungeon_master("I enter the dark forest and look around", player_state)
print(json.dumps(result, indent=2))

In [ ]:
result2 = dungeon_master("I follow the path deeper into the forest", player_state)
print(json.dumps(result2, indent=2))

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from groq import Groq
import json
import re
import os
from pyngrok import ngrok

app = FastAPI()
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# Game state
player_state = {
    "health": 100,
    "inventory": ["torch", "dagger"],
    "quest": "Find the lost sword of Eldoria",
    "location": "village entrance"
}

conversation_history = []

system_prompt = """You are an AI Dungeon Master running a fantasy RPG.
You MUST respond ONLY in valid JSON format like this:
{
    "narrative": "description of what happens",
    "npc_dialogue": "any NPC speech or empty string",
    "player_state": {
        "health": <number 0-100>,
        "inventory": ["item1", "item2"],
        "quest": "current quest",
        "location": "current location"
    },
    "choices": ["option 1", "option 2", "option 3"]
}
Never respond with plain text. Always valid JSON only."""

class PlayerAction(BaseModel):
    action: str

@app.get("/")
def home():
    return {"message": "AI Dungeon Master API is running!"}

@app.get("/state")
def get_state():
    return player_state

@app.post("/action")
def take_action(body: PlayerAction):
    conversation_history.append({"role": "user", "content": body.action})

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Player state: {json.dumps(player_state)}"},
            *conversation_history[-6:]
        ],
        temperature=0.7,
        max_tokens=500
    )

    raw = response.choices[0].message.content.strip()
    conversation_history.append({"role": "assistant", "content": raw})

    try:
        json_match = re.search(r'\{.*\}', raw, re.DOTALL)
        if json_match:
            result = json.loads(json_match.group())
            if "player_state" in result:
                player_state.update(result["player_state"])
            return result
        else:
            return {"narrative": raw, "choices": [], "player_state": player_state}
    except:
        return {"narrative": raw, "choices": [], "player_state": player_state}

@app.post("/reset")
def reset_game():
    global player_state, conversation_history
    player_state = {
        "health": 100,
        "inventory": ["torch", "dagger"],
        "quest": "Find the lost sword of Eldoria",
        "location": "village entrance"
    }
    conversation_history = []
    return {"message": "Game reset!"}

# Start server
import uvicorn
import threading

ngrok.set_auth_token("YOUR_NGROK_TOKEN")
public_url = ngrok.connect(8000)
print(f"\n=== API Server Running ===")
print(f"Public URL: {public_url}")
print(f"Test it: {public_url}/docs\n")

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run)
thread.start()